In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn import metrics

In [ ]:
# loading checked csvs
df_query1 = pd.read_csv("data/article_samples/q1_samples_checked.csv")
df_query2 = pd.read_csv("data/article_samples/q2_samples_checked.csv")
df_query3 = pd.read_csv("data/article_samples/q3_samples_checked.csv")

# specifying query source
df_query1["query"] = "query1"
df_query2["query"] = "query2"
df_query3["query"] = "query3"

# combining query dfs and specifying data types
all_queries = pd.concat([df_query1, df_query2, df_query3], ignore_index=True)
all_queries['is-thematically-relevant'] = all_queries['is-thematically-relevant'].astype(bool)
all_queries['has-event'] = all_queries['has-event'].astype(bool)
all_queries['has-named-location'] = all_queries['has-named-location'].astype(bool)
all_queries['has-space-type'] = all_queries['has-space-type'].astype(bool)
all_queries['query'] = all_queries['query'].astype('category')

In [ ]:
all_queries.head()

In [ ]:
bool_cols = ["is-thematically-relevant", "has-event", "has-named-location", "has-space-type"]
relevance_distribution = all_queries.groupby(["query", "decile"])[bool_cols].mean().reset_index()
relevance_distribution


In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=relevance_distribution, x="decile", y="is-thematically-relevant", hue="query", marker="o")
plt.title("Relevance Distribution by Decile for Each Query Strategy")
plt.xlabel("Decile (10 = Most Similar)")
plt.ylabel("Fraction Relevant")
plt.xticks(range(1, 11))
plt.grid(True)
plt.gca().invert_xaxis()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=relevance_distribution, x="decile", y="has-event", hue="query", marker="o")
plt.title("Distribution of Event-mentioning Articles by Decile for Each Query Strategy")
plt.xlabel("Decile (10 = Most Similar)")
plt.ylabel("Fraction Relevant")
plt.xticks(range(1, 11))
plt.grid(True)
plt.gca().invert_xaxis()
plt.show()


In [ ]:
# calculating AUC

auc_scores = {}
for query in all_queries["query"].unique():
    query_subset = all_queries[all_queries["query"] == query]
    grouped = query_subset.groupby("decile")["is-thematically-relevant"].mean()
    auc = metrics.auc(grouped.index, grouped.values)
    auc_scores[query] = auc

print(auc_scores)


In [ ]:
##### I HAVE DECIDED TO GO WITH QUERY 1! #####
# NOTE: the below 3 cells (cut-off subsetting, sampling, saving) have been done in lightning studio because the corpus data is read in and stored there (notebook: synth-queries)

In [ ]:
# # reading in articles, scored (cosine similarity) against query 1 
# scored_articles = pd.read_csv("data/scored_articles.csv")

# scored_articles.head()

In [ ]:
# # subsetting based on cut-off threshold

# percentile = 0.9  
# threshold = scored_articles["score"].quantile(percentile)
# articles_subset = scored_articles[scored_articles["score"] >= threshold]

# articles_subset

In [ ]:
# # sampling 100 for 2nd verification
# subset_samples = articles_subset.sample(n=100, 
#                                         random_state=123).sort_values('score', ascending=False)

# # saving file to inspect
# subset_samples.to_csv('data/subset_samples.csv', index=False, encoding="utf-8-sig")

In [ ]:
# having checked my samples manually, i'm loading the csv here to interpret it

In [ ]:
subset_samples_checked = pd.read_csv('data/subset_samples_checked.csv')
subset_samples_checked.info()
subset_samples_checked

In [ ]:
# function to check the number of rows for which a combination of has-event/has-location/has-space-type is true

def true_check(df, cols):
    # where 'cols' is a list of boolean columns to check
    true_count = subset_samples_checked[cols].all(axis=1).sum()
    print(f'No. of rows where {cols} is/are True: {true_count}')

true_check(subset_samples_checked, ['has-event'])